In [14]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report

from imblearn.over_sampling import SMOTE

print("\n✅ Libraries Imported")

# =========================
# 2. LOAD DATASET
# =========================
df = pd.read_csv("Crop_recommendation.csv")

print("\n✅ Dataset Loaded:", df.shape)

# =========================
# 3. PREPROCESSING
# =========================
df.rename(columns={
    'temperature': 'temp',
    'label': 'crop'
}, inplace=True)

# Create soil_type from Nitrogen
df['soil_type'] = pd.cut(df['N'], bins=3, labels=['sandy','loamy','clay'])

# Keep NPK also
df = df[['N','P','K','temp','humidity','rainfall','soil_type','crop']]

# Encode categorical
df['soil_type'] = df['soil_type'].astype('category').cat.codes
df['crop'] = df['crop'].astype('category').cat.codes

print("\n✅ Final Dataset Ready")
print(df.head())

# =========================
# 4. FEATURES
# =========================
X = df.drop('crop', axis=1)
y = df['crop']

# =========================
# 5. SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# 6. SCALING
# =========================
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# 7. SMOTE
# =========================
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# =========================
# 8. MODELS
# =========================

rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train_sm, y_train_sm)
rf_pred = rf.predict(X_test)

gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

lr = LogisticRegression(max_iter=1000)
bag_lr = BaggingClassifier(estimator=lr, n_estimators=10, random_state=42)
bag_lr.fit(X_train, y_train)
lr_pred = bag_lr.predict(X_test)

svm = SVC()
bag_svm = BaggingClassifier(estimator=svm, n_estimators=10, random_state=42)
bag_svm.fit(X_train, y_train)
svm_pred = bag_svm.predict(X_test)

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

# =========================
# 9. EVALUATION
# =========================
print("\n===== 📊 F1 SCORES =====")
print("RF:", f1_score(y_test, rf_pred, average='weighted'))
print("GB:", f1_score(y_test, gb_pred, average='weighted'))
print("LR:", f1_score(y_test, lr_pred, average='weighted'))
print("SVM:", f1_score(y_test, svm_pred, average='weighted'))
print("Dummy:", f1_score(y_test, dummy_pred, average='weighted'))

print("\n===== REPORT (RF) =====")
print(classification_report(y_test, rf_pred))

# =========================
# 10. TUNING
# =========================
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, scoring='f1_weighted', cv=3)
grid.fit(X_train, y_train)

print("\nBest Params:", grid.best_params_)


✅ Libraries Imported

✅ Dataset Loaded: (2200, 8)

✅ Final Dataset Ready
    N   P   K       temp   humidity    rainfall  soil_type  crop
0  90  42  43  20.879744  82.002744  202.935536          1    20
1  85  58  41  21.770462  80.319644  226.655537          1    20
2  60  55  44  23.004459  82.320763  263.964248          1    20
3  74  35  40  26.491096  80.158363  242.864034          1    20
4  78  42  42  20.130175  81.604873  262.717340          1    20

===== 📊 F1 SCORES =====
RF: 0.990894856106666
GB: 0.9863222244532512
LR: 0.927754500780196
SVM: 0.972482500254119
Dummy: 0.003952569169960474

===== REPORT (RF) =====
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        20
           1       1.00      1.00      1.00        20
           2       1.00      0.95      0.97        20
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        20
           5       1.00      1.00      1.00